# Building the Gold Layer: Aggregations, Hierarchies, and Current State

The gold layer is where raw IoT events become business value. This notebook builds three gold tables that cover the main patterns you'll encounter in production: time-window aggregations, rollups over an asset hierarchy, and a current-device-state table. Along the way, we'll look at a subtle but important trap: incorrect aggregation across multiple steps.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import json
import shutil
from pathlib import Path
import pyarrow as pa
import pandas as pd
from pyiceberg.catalog.sql import SqlCatalog
from pyspark.sql import functions as F
from pyspark.sql.window import Window

sys.path.insert(0, str(Path.cwd()))

In [ ]:
# Clean up from previous runs
for d in ["../data/warehouse_silver", "../data/warehouse_gold"]:
    shutil.rmtree(d, ignore_errors=True)

catalog = SqlCatalog("silver", **{
    "uri": "sqlite:///../data/warehouse_silver/silver_catalog.db",
    "warehouse": "file://" + str(Path("../data/warehouse_silver").resolve()),
})
catalog.create_namespace_if_not_exists("silver")

events = [json.loads(l) for l in open("../data/input/events.jsonl")]
df_events = pd.DataFrame(events)
arrow_table = pa.Table.from_pandas(df_events)
silver_events = catalog.create_table("silver.events", schema=arrow_table.schema)
silver_events.append(arrow_table)
print(f"Silver events: {len(events):,} rows")

In [ ]:
from helpers import get_spark, inspect_gold_table

spark = get_spark("GoldLayerPatterns", gold_warehouse="../data/warehouse_gold")
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.gold")

silver_df = spark.createDataFrame(silver_events.scan().to_arrow().to_pandas())
silver_df = silver_df.withColumn("event_ts", F.to_timestamp("time"))
print(f"Silver rows: {silver_df.count():,}")

## Pattern 1: Time-Window Aggregations

The simplest gold pattern — aggregate events into time buckets (hours, days). This answers "how active were our devices over time?" Use cases: capacity planning, anomaly detection, operational dashboards.

In [ ]:
# Gold: hourly event counts per event type
hourly_by_type = silver_df.groupBy(
    F.date_trunc("hour", "event_ts").alias("event_hour"),
    "type"
).agg(
    F.count("*").alias("event_count"),
    F.countDistinct("source").alias("active_devices")
).orderBy("event_hour", "type")

hourly_by_type.writeTo("local.gold.hourly_by_type").createOrReplace()
print(f"Hourly gold rows: {hourly_by_type.count():,}")
spark.sql("SELECT * FROM local.gold.hourly_by_type ORDER BY event_hour DESC, event_count DESC LIMIT 10").show()

In [ ]:
# Rolling 7-day distinct active devices per event type
# Step 1: daily counts as foundation
daily_counts = silver_df.groupBy(
    F.to_date("event_ts").alias("event_date"),
    "type"
).agg(F.countDistinct("source").alias("active_devices"))

# Step 2: rolling 7-day window
# rangeBetween requires a numeric ordering column. Convert date to unix days.
w = Window.partitionBy("type").orderBy(F.unix_date("event_date")).rangeBetween(-6, 0)

rolling = daily_counts.withColumn(
    "rolling_7d_devices",
    F.sum("active_devices").over(w)  # sum over daily counts — see note below
).orderBy("event_date", "type")

rolling.writeTo("local.gold.rolling_7d_activity").createOrReplace()
rolling.show(10)

### A note on rolling distinct counts

`rolling_7d_devices` is the SUM of daily distinct device counts over 7 days — not the true distinct count across the 7-day window. If device A appeared on day 1 and day 3, it's counted twice. Getting the true rolling distinct count requires either re-evaluating the full 7-day window from raw data or using approximate counting (HyperLogLog). For most operational dashboards, summing daily distinct counts is an acceptable approximation — but document the assumption.

## Pattern 2: Aggregation Over the Asset Hierarchy

The asset hierarchy from `cmdata.jsonl`: devices belong to groups (`c8y_DeviceSubgroup`), groups can have subgroups. The business wants event counts rolled up to the top-level group. Three rules apply:

- **Additive measures** (event counts, durations): safe to SUM at every hierarchy level
- **Semi-additive measures** (distinct device counts): must be re-evaluated at each level, not summed from child counts
- **Non-additive measures** (rates, averages): must carry numerator and denominator separately, compute rate only at query time

In [ ]:
# Load and flatten the hierarchy
devices = [json.loads(l) for l in open("../data/input/cmdata.jsonl")]

# Build parent→child relationships
parent_child = []
for d in devices:
    parent_id = str(d["id"])
    parent_name = d.get("name", parent_id)
    parent_type = d.get("type", "unknown")
    for child_id in d.get("childAssets", []) + d.get("childDevices", []):
        parent_child.append({
            "child_id": str(child_id),
            "parent_id": parent_id,
            "parent_name": parent_name,
            "parent_type": parent_type,
        })

hierarchy_df = spark.createDataFrame(parent_child).dropDuplicates(["child_id"])
print(f"Hierarchy relationships: {hierarchy_df.count():,}")
hierarchy_df.show(5)

In [ ]:
# Join silver events with the hierarchy
enriched = silver_df.join(
    hierarchy_df.select(
        F.col("child_id").alias("source"),
        F.col("parent_id").alias("group_id"),
        F.col("parent_name").alias("group_name"),
    ),
    on="source",
    how="left"
).withColumn("group_name", F.coalesce("group_name", F.lit("ungrouped")))

# Gold: daily event counts per group (additive — safe to SUM)
group_daily = enriched.groupBy(
    F.to_date("event_ts").alias("event_date"),
    "group_id",
    "group_name",
).agg(
    F.count("*").alias("event_count"),                # additive
    F.countDistinct("source").alias("device_count"),  # semi-additive — re-evaluated at group level
    # Keep alarm events separately for rate computation
    F.sum(F.when(F.col("type").contains("Alarm"), 1).otherwise(0)).alias("alarm_event_count"),
).orderBy("event_date", "group_name")

group_daily.writeTo("local.gold.group_daily").createOrReplace()
print(f"Group daily gold rows: {group_daily.count():,}")
group_daily.show(10)

## The Correct Aggregation Trap: Why Averaging Averages Is Wrong

Suppose you have two device groups and store per-group average events per hour:
- Group A: 3 devices, average 100 events/hr
- Group B: 7 devices, average 20 events/hr

**Wrong rollup:** (100 + 20) / 2 = **60 events/hr**
**Correct rollup:** (3 × 100 + 7 × 20) / (3 + 7) = 440 / 10 = **44 events/hr**

The error is 36%. It gets worse with more imbalanced groups.

**The fix:** never store bare averages in gold tables. Store `sum` and `count` (numerator and denominator) separately. Compute the ratio only at query time.

| Measure | Type | Correct rollup |
|---|---|---|
| Event count | Additive | SUM at every level — always safe |
| Total duration | Additive | SUM at every level |
| Average response time | Non-additive | SUM(duration) / SUM(count) |
| Alarm rate (%) | Non-additive | SUM(alarm_count) / SUM(total_count) |
| Distinct active devices | Semi-additive | COUNT(DISTINCT) re-evaluated at each level |
| Current device status | Non-additive | MAX or LATEST per device |

**Additional gotchas:**

- **NULL arithmetic**: `NULL + x = NULL` in SQL. Use `COALESCE(x, 0)` for counts, but beware of masking real NULLs in averages.
- **Integer overflow**: COUNT(*) on billions of rows can overflow INT32. PySpark uses Long (BIGINT) by default — verify if downstream systems expect int.
- **Time zone and DST**: Aggregate in UTC. A DST transition at 02:00 can make an "hourly" window span 0 or 2 hours in local time.
- **Duplicate events**: IoT devices retry on failure. Deduplicate on the event `id` field before aggregating.

In [ ]:
# Demonstrate: per-group averages stored separately, then rolled up naively
from pyspark.sql import Row

# Simulate pre-aggregated per-group averages (as you might find in a poorly-designed gold table)
bad_gold = spark.createDataFrame([
    Row(group="Group A", device_count=3, avg_events_per_hr=100.0),
    Row(group="Group B", device_count=7, avg_events_per_hr=20.0),
])

# Wrong rollup: simple average of averages
bad_rollup = bad_gold.agg(F.avg("avg_events_per_hr").alias("wrong_average"))
print("Wrong rollup (average of averages):")
bad_rollup.show()

# Correct rollup: weighted average using numerator/denominator
good_gold = spark.createDataFrame([
    Row(group="Group A", device_count=3, total_events=300.0),  # 3 × 100
    Row(group="Group B", device_count=7, total_events=140.0),  # 7 × 20
])

good_rollup = good_gold.agg(
    (F.sum("total_events") / F.sum("device_count")).alias("correct_average")
)
print("Correct rollup (weighted average):")
good_rollup.show()

In [ ]:
# Alarm rate: always carry numerator (alarm_event_count) and denominator (event_count) separately.
# Never store alarm_rate directly in gold — compute it at query time.
print("Alarm rates per group (computed at query time from stored counts):")
spark.sql("""
    SELECT
        group_name,
        SUM(event_count)       AS total_events,
        SUM(alarm_event_count) AS total_alarms,
        ROUND(100.0 * SUM(alarm_event_count) / NULLIF(SUM(event_count), 0), 2) AS alarm_rate_pct
    FROM local.gold.group_daily
    GROUP BY group_name
    ORDER BY total_events DESC
    LIMIT 10
""").show(truncate=False)

## Pattern 3: Current Device State (SCD Type 1)

"Current state" gold tables hold one row per device, showing the latest known state.

- **SCD Type 1** (Slowly Changing Dimension Type 1): overwrite on update. No history kept.
- **Use case**: device dashboards, "is device X healthy right now?"
- **Implementation**: Iceberg's `MERGE INTO` — Spark Iceberg extension.
- `MERGE INTO` checks each incoming row against the existing table: if a matching device already exists, UPDATE it; otherwise INSERT.

In [ ]:
# Build: latest event per device (type, time)
from pyspark.sql.window import Window

device_window = Window.partitionBy("source").orderBy(F.col("event_ts").desc())
latest_events = silver_df.withColumn("rn", F.row_number().over(device_window)).filter(F.col("rn") == 1).drop("rn")

# Current state: device_id, latest_event_type, latest_event_time
current_state = latest_events.select(
    F.col("source").alias("device_id"),
    F.col("type").alias("latest_event_type"),
    F.col("event_ts").alias("last_seen"),
).withColumn("updated_at", F.current_timestamp())

# Write initial state
current_state.writeTo("local.gold.device_current_state").createOrReplace()
print(f"Devices in gold: {current_state.count():,}")
current_state.show(5)

In [ ]:
# Simulate new events arriving for a subset of devices
from datetime import datetime, timezone

new_events = spark.createDataFrame([
    ("140673", "MaintenanceMode", datetime(2024, 9, 1, 12, 0, tzinfo=timezone.utc)),
    ("140675", "AlarmCleared",    datetime(2024, 9, 1, 12, 5, tzinfo=timezone.utc)),
    ("999001", "NewDevice",       datetime(2024, 9, 1, 12, 1, tzinfo=timezone.utc)),  # new device
], ["device_id", "latest_event_type", "last_seen"])

new_events = new_events.withColumn("updated_at", F.current_timestamp())

# Register as temp view for MERGE INTO
new_events.createOrReplaceTempView("new_state")

# MERGE INTO: SCD Type 1 — update existing rows, insert new ones
spark.sql("""
    MERGE INTO local.gold.device_current_state AS target
    USING new_state AS source
    ON target.device_id = source.device_id
    WHEN MATCHED THEN UPDATE SET
        target.latest_event_type = source.latest_event_type,
        target.last_seen         = source.last_seen,
        target.updated_at        = source.updated_at
    WHEN NOT MATCHED THEN INSERT *
""")

print("After MERGE INTO:")
spark.sql("SELECT * FROM local.gold.device_current_state WHERE device_id IN ('140673','140675','999001')").show()

In [ ]:
# Every MERGE INTO is an atomic Iceberg snapshot
spark.sql("SELECT snapshot_id, made_current_at, is_current_ancestor FROM local.gold.device_current_state.history").show()

## Review Questions

1. You have a gold table with `avg_response_ms` per device per hour. You need the fleet-wide average. Why is `AVG(avg_response_ms)` wrong? What columns would you need to store instead?
2. An IoT device sends the same event twice with the same `id`. If you COUNT(*) before deduplication, what happens to your `alarm_rate_pct`? How would you fix the upstream pipeline?
3. When rolling device event counts up from the device level to the group level, is COUNT(DISTINCT device_id) additive? What goes wrong if you SUM the per-group distinct counts?
4. What is the difference between SCD Type 1 and SCD Type 2? Give an example IoT use case for each.
5. A DST transition at 02:00 creates an hour with 0 or 2 occurrences in local time. How does storing and aggregating in UTC prevent this?

## Challenges

- Build a "top-10 most active device groups per day" gold table using the `group_daily` table.
- Prove the averaging-averages bug: create a gold table that stores per-device avg events/hr; roll it up; show the wrong answer; fix it by carrying sum and count.
- Implement SCD Type 2: add `effective_from` and `effective_to` columns to the device state table. On update, close the old row and insert a new one. Hint: use `MERGE INTO` with `WHEN MATCHED THEN UPDATE` to set `effective_to`, and a second INSERT for the new row.
- Deduplicate `events.jsonl` on `id` before aggregating and measure how much the alarm rate changes.

## Summary

Key takeaways:

- **Time-window aggregations** (hourly/daily counts) are the most common gold pattern. They are additive: safe to SUM across groups.
- **Asset hierarchy rollups** require understanding which measures are additive (event counts), semi-additive (distinct devices), or non-additive (rates, averages).
- **Never store bare averages** in gold tables. Store numerator and denominator separately; compute the ratio at query time.
- **SCD Type 1** (`MERGE INTO`) provides a current-state snapshot. Iceberg's atomic snapshots make MERGE INTO safe: the old state remains readable until the merge commits.
- Watch for NULL arithmetic, integer overflow, duplicate IoT events, and time-zone drift when building gold tables.
- **Next**: making gold updates efficient with incremental processing and Iceberg snapshots as checkpoints.